# Exercise 05: Hyperparameter Tuning (Optuna)

<img src="images_optuna/optuna.png" width="800">

- Hyperparameter tuning tools: [Optuna](https://optuna.org) or [Ray Tune](https://docs.ray.io/en/latest/tune/index.html)
- Recommended reading: [8 Optuna + Ray Tune Playbooks for Fast Hyperparameter Search](https://medium.com/@ThinkingLoop/8-optuna-ray-tune-playbooks-for-fast-hyperparameter-search-4fea9c1f6791)


## Grid Search vs. Optimization Frameworks
**Why hyperparameter tuning?**
Hyperparameters (learning rate, model width, dropout, etc.) strongly affect performance.

**What is grid search?**  Grid search picks a fixed set of values for each hyperparameter and tries every possible combination. It is simple and reproducible, but can become very expensive as the number of hyperparameters grows.

**Why use frameworks like Optuna or Ray Tune here?**
- in LLM training, there are several tunable parameters (e.g., learning rate, dropout, model width), so exhaustive combinations quickly explode.
- Frameworks use smarter search strategies to focus on promising regions instead of testing everything blindly.
- They support early stopping/pruning, so weak trials can end early and save compute.
- They track trial results cleanly and make it easier to compare runs and reproduce outcomes.

**Notebook overview**
- Build a tiny dataset
- Train a tiny Transformer for a few steps
- Let Optuna search over hyperparameters
- Open the Optuna dashboard in your browser


In [ ]:
# %pip install -q optuna optuna-dashboard plotly torch nbformat scikit-learn

In [1]:
import random
from pathlib import Path

import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cpu


## Tiny character dataset


In [2]:
base_text = (
    'machine learning models depend on hyperparameters. '
    'good tuning improves validation loss. '
    'transformers can be small and still useful for teaching. '
)
text = (base_text * 300).lower()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print('text length:', len(data), 'vocab size:', vocab_size)

text length: 43800 vocab size: 22


In [3]:
def get_batch(split, batch_size, block_size):
    src = train_data if split == 'train' else val_data
    ix = torch.randint(len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i:i+block_size] for i in ix])
    y = torch.stack([src[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

## Tiny Transformer LM


In [4]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(block_size, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok(idx) + self.pos(pos)[None, :, :]

        mask = torch.triu(torch.ones(T, T, device=idx.device), diagonal=1).bool()
        x = self.encoder(x, mask=mask)
        x = self.ln(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss


## How an Optuna Optimization Run Works
Optuna performs hyperparameter optimization by repeatedly executing an objective function with different hyperparameter configurations. Each execution of the objective function is called a trial. Each trial samples hyperparameters, trains the model, and evaluates its performance.

```
Study
 ├─ Trial 1 → parameters → train model → loss
 ├─ Trial 2 → parameters → train model → loss
 ├─ Trial 3 → parameters → train model → loss
 └─ ...
```


## Optuna objective

What Optuna should optimize. Inside this function we:
	•	sample hyperparameters
	•	train the model
	•	return the evaluation metric

We tune:
- `lr` (learning rate)
- `d_model`
- `n_heads`
- `n_layers`
- `dropout`
- `batch_size`

We train only briefly (`train_steps`) so many trials can run quickly.


The Search Space / Search Ranges are defined implicitly inside the objective function using: trial.suggest_*

In [5]:
block_size = 64
train_steps = 120
eval_steps = 30

def evaluate(model, batch_size):
    model.eval()
    losses = []
    with torch.no_grad():
        for _ in range(eval_steps):
            xb, yb = get_batch('val', batch_size, block_size)
            _, loss = model(xb, yb)
            losses.append(loss.item())
    return float(np.mean(losses))

def objective(trial):
    # 1) Define search space (what Optuna is allowed to tune).
    d_model = trial.suggest_categorical('d_model', [32, 64, 96])
    n_heads = trial.suggest_categorical('n_heads', [1, 2, 4])
    n_layers = trial.suggest_int('n_layers', 1, 3)
    dropout = trial.suggest_float('dropout', 0.0, 0.3)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    # 2) Prune invalid configurations early.
    if d_model % n_heads != 0:
        raise optuna.TrialPruned()

    model = TinyTransformerLM(
        vocab_size=vocab_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        block_size=block_size,
        dropout=dropout,
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    # 3) Short training loop for this trial.
    for step in range(train_steps):
        xb, yb = get_batch('train', batch_size, block_size)
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        # 4) Periodically report intermediate val loss so the pruner can stop weak trials.
        if step % 20 == 0:
            val_loss = evaluate(model, batch_size)
            trial.report(val_loss, step)
            if trial.should_prune():
                raise optuna.TrialPruned()

    # 5) Return final objective value (Optuna minimizes this).
    return evaluate(model, batch_size)


## Define a study that manages the optimization process

```
Study: 'tiny_transformer_lm'
 ├─ Trial 1 → parameters → train model → loss
 ├─ Trial 2 → parameters → train model → loss
 ├─ Trial 3 → parameters → train model → loss
 └─ ...
```

In [6]:
# Persistent storage lets Optuna dashboard read all trials from disk.
DB_PATH = Path('ex05_hyperparam_tinylm.db')
STORAGE = f'sqlite:///{DB_PATH}'
STUDY_NAME = 'tiny_transformer_lm'

# load_if_exists=True means reruns append trials to the same study.
study = optuna.create_study(
    direction='minimize',
    study_name=STUDY_NAME,
    storage=STORAGE,
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=20),
)

# Increase n_trials for better search quality (at the cost of runtime).
study.optimize(objective, n_trials=20, show_progress_bar=True)

print('Best value (val loss):', study.best_value)
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

print('\nDB saved at:', DB_PATH.resolve())
print('Dashboard storage URL:', f"sqlite:///{DB_PATH.resolve()}")

[I 2026-03-16 16:09:03,964] Using an existing study with name 'tiny_transformer_lm' instead of creating a new one.


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-16 16:09:06,907] Trial 40 pruned. 
[I 2026-03-16 16:09:17,640] Trial 41 finished with value: 0.04754134813944499 and parameters: {'d_model': 96, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.08159994998137866, 'lr': 0.004506146886490431, 'batch_size': 64}. Best is trial 39 with value: 0.04528191598753135.
[I 2026-03-16 16:09:28,773] Trial 42 finished with value: 0.04368143118917942 and parameters: {'d_model': 96, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.07399875137006998, 'lr': 0.0049803186156717255, 'batch_size': 64}. Best is trial 42 with value: 0.04368143118917942.
[I 2026-03-16 16:09:40,501] Trial 43 finished with value: 0.04311741503576438 and parameters: {'d_model': 96, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.08045150680823032, 'lr': 0.00464754601315652, 'batch_size': 64}. Best is trial 43 with value: 0.04311741503576438.


/var/folders/4x/zyqt0lhj12n2r1v2pq_hyjpw0000gn/T/ipykernel_66420/2400198072.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)


[I 2026-03-16 16:09:50,436] Trial 44 finished with value: 0.04680119678378105 and parameters: {'d_model': 96, 'n_heads': 1, 'n_layers': 2, 'dropout': 0.07998923567760438, 'lr': 0.004719435615499848, 'batch_size': 64}. Best is trial 43 with value: 0.04311741503576438.
[I 2026-03-16 16:10:01,466] Trial 45 finished with value: 0.04759671377638976 and parameters: {'d_model': 96, 'n_heads': 1, 'n_layers': 2, 'dropout': 0.07071702957486839, 'lr': 0.004830573537515186, 'batch_size': 64}. Best is trial 43 with value: 0.04311741503576438.
[I 2026-03-16 16:10:08,594] Trial 46 finished with value: 0.05410596989095211 and parameters: {'d_model': 96, 'n_heads': 1, 'n_layers': 2, 'dropout': 0.11697502669932863, 'lr': 0.004306221418055612, 'batch_size': 32}. Best is trial 43 with value: 0.04311741503576438.
[I 2026-03-16 16:10:18,592] Trial 47 finished with value: 0.04491568468511105 and parameters: {'d_model': 96, 'n_heads': 1, 'n_layers': 2, 'dropout': 0.09396017624951364, 'lr': 0.00492863620532530

## Database

- Keeps all Optuna trial results in one place so progress is not lost between runs.
- Lets us resume the study later and continue searching from existing trials.
- Makes results easy to inspect, compare, and visualize (e.g., with `optuna-dashboard`).
- Supports reproducibility by storing parameters, metrics, and trial metadata.

In [8]:
print(DB_PATH.resolve())

/Users/Lorena/Developer/nlp-llm/FS2026_nlp-llm/exercises/EX05/ex05_hyperparam_tinylm.db


## Optuna dashboard (browser)

Run this in a terminal from the notebook directory:

```bash
optuna-dashboard sqlite:///ex05_hyperparam_tinylm.db --host 127.0.0.1 --port 8080
```

Then open:

```text
http://127.0.0.1:8080
```

Inspect in the dashboard:
- History: whether objective values decrease over trial number.
- Importance: in this run, `lr` and `dropout` dominate; `n_layers` is smaller.
- Parallel/Slice: how combinations like (`lr`, `dropout`) relate to objective.
- Trials table/details: compare completed vs pruned trials and inspect the best trial.

## Dashboard Walkthrough

<img src="images_optuna/Dashboard.png" width="800">
- Optuna Dashboard page: starting screen where all optimization studies can be viewed and managed (STUDY_NAME = 'tiny_transformer_lm')

<img src="images_optuna/Trials.png" width="800">
- Trial Table overview

<img src="images_optuna/TrialHistory.png" width="1000">
- Trial History: improvement during optimization, finding better trials over time.

<img src="images_optuna/IntermediateValues.png" width="1000">
- Intermediate Values: Training progress per trial over steps. Bad trials diverge early -> pruned.

<img src="images_optuna/Hyperparam.png" width="1000">
- Hyperparameter Importance: which parameters matter the most? -> learning rate and dropout have the strongest influence on model performance.

<img src="images_optuna/ParallelCoordinate.png" width="1000">
- Parallel Coordinate Plot: show interaction between hyperparameters.

<img src="images_optuna/singleparameffect.png" width="800">
- how a single parameter affects performance.

<div style="display:flex; gap:16px; align-items:flex-start; margin-top:10px;">
  <img src="images_optuna/besttrial.png" width="480">
  <img src="images_optuna/trial43.png" width="600">
</div>
- Best Trial summary:  final optimized configuration.

